# Detección y subtipificación de leucemia linfoblástica aguda usando inteligencia artificial

**Proyecto final unificado - Curso de Inteligencia Artificial**

Este notebook integra, depura y resume los cuatro notebooks desarrollados durante el proyecto sobre el dataset **Acute Lymphoblastic Leukemia (ALL) image dataset** de Kaggle (`mehradaria/leukemia`). El problema consiste en analizar imágenes microscópicas de frotis de sangre periférica para apoyar la detección y subtipificación de leucemia linfoblástica aguda (LLA) en cuatro clases:

- `Benign`
- `Early`
- `Pre`
- `Pro`

La leucemia linfoblástica aguda es una enfermedad hematológica en la que la identificación temprana y la caracterización del subtipo son relevantes para orientar estudios diagnósticos y decisiones clínicas. En la práctica médica, el diagnóstico definitivo no depende solo de la imagen microscópica: debe confirmarse con pruebas clínicas y de laboratorio, especialmente citometría de flujo, además de la evaluación del especialista.

El objetivo de este proyecto no es reemplazar al diagnóstico clínico, sino evaluar un pipeline académico de IA aplicado a imágenes médicas: primero se realiza análisis exploratorio y estadístico, luego se entrenan modelos supervisados clásicos, después una red neuronal profunda densa sobre variables extraídas y finalmente se explora la estructura latente mediante aprendizaje no supervisado.

> Este notebook no es una acumulación de código: es una versión final, curada, explicativa y defendible académicamente.


## 1. Descripción del dataset

El dataset usado fue publicado en Kaggle como **Acute Lymphoblastic Leukemia (ALL) image dataset**:

- Fuente: [Kaggle - mehradaria/leukemia](https://www.kaggle.com/datasets/mehradaria/leukemia/data?select=Segmented)
- Repositorio asociado: [MehradAria/ALL-Subtype-Classification](https://github.com/MehradAria/ALL-Subtype-Classification)
- Paper asociado: *A Fast and Efficient CNN Model for B-ALL Diagnosis and its Subtypes Classification using Peripheral Blood Smear Images*

El conjunto contiene imágenes microscópicas de células obtenidas a 100x, con dos representaciones principales:

- **Originales:** imágenes microscópicas del frotis de sangre periférica.
- **Segmentadas:** imágenes donde la célula de interés está separada del fondo, útiles para estimar variables de región, forma y textura.

En esta copia del proyecto, el archivo de características generado previamente contiene **3256 muestras**. La distribución observada es:

| Clase | Imágenes |
|---|---:|
| Benign | 504 |
| Early | 985 |
| Pre | 963 |
| Pro | 804 |

El dataset es útil para un proyecto de IA porque permite combinar visión por computador, estadística, aprendizaje supervisado, redes neuronales densas y aprendizaje no supervisado dentro de un caso biomédico realista.


## 2. Librerías y configuración

Se centralizan los imports para evitar repeticiones. TensorFlow se importa solo en la sección de DNN si se decide reentrenar el modelo.


In [ ]:
import os
import re
import math
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None

from IPython.display import display, Markdown, Image as IPImage

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
np.random.seed(SEED)

CLASS_ORDER = ["Benign", "Early", "Pre", "Pro"]
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


## 3. Carga robusta de archivos generados

Los notebooks originales fueron ejecutados en ambientes distintos. Por eso los archivos pueden aparecer con sufijos como `(1)` o estar dentro de carpetas de resultados. La función siguiente busca por nombre normalizado y evita depender de rutas absolutas personales.


In [ ]:
BASE_DIR = Path(os.getenv("LEUKEMIA_PROJECT_DIR", Path.cwd())).resolve()

candidate_dirs = [
    BASE_DIR,
    BASE_DIR / "unsupervised_outputs",
    BASE_DIR / "outputs" / "final_notebook",
    BASE_DIR / "dnn_outputs",
    Path("/content/drive/MyDrive/Proyecto_IA/parte2"),
    Path("/content/drive/MyDrive/Proyecto_IA/parte2/unsupervised_outputs"),
    Path("/content/drive/MyDrive/Proyecto_IA/parte2/dnn_outputs"),
]

SEARCH_DIRS = []
for directory in candidate_dirs:
    try:
        if directory.exists() and directory not in SEARCH_DIRS:
            SEARCH_DIRS.append(directory)
    except Exception:
        pass


def normalize_filename(name):
    path = Path(str(name))
    stem = re.sub(r"\s*\(\d+\)$", "", path.stem.strip())
    return f"{stem.lower()}{path.suffix.lower()}"


def find_file(possible_names):
    if isinstance(possible_names, str):
        possible_names = [possible_names]

    normalized_targets = {normalize_filename(name) for name in possible_names}

    for directory in SEARCH_DIRS:
        for name in possible_names:
            direct = directory / name
            if direct.exists():
                return direct

    for directory in SEARCH_DIRS:
        try:
            for path in directory.rglob("*"):
                if path.is_file() and normalize_filename(path.name) in normalized_targets:
                    return path
        except Exception:
            continue
    return None


FILES = {
    "features": find_file("leukemia_handcrafted_features.csv"),
    "ranking": find_file("leukemia_feature_ranking.csv"),
    "model_results": find_file("model_comparison_results.csv"),
    "kfold_top100": find_file("kfold10_results_top100.csv"),
    "dnn_features": find_file("dnn_features_results.csv"),
    "dnn_final": find_file("dnn_final_comparison.csv"),
    "clustering": find_file("clustering_comparison.csv"),
    "final_comparison": find_file("final_comparison_all_models.csv"),
}

print("Directorio base:", BASE_DIR)
print("Directorios de búsqueda:")
for directory in SEARCH_DIRS:
    print(" -", directory)

print("\nArchivos detectados:")
for label, path in FILES.items():
    print(f"{label:18s} -> {path if path is not None else 'No encontrado'}")


In [ ]:
def read_csv_if_exists(path, label):
    if path is None:
        print(f"[Aviso] No se encontró: {label}")
        return None
    df = pd.read_csv(path)
    print(f"{label}: {df.shape} <- {path.name}")
    return df


df = read_csv_if_exists(FILES["features"], "features")
ranking = read_csv_if_exists(FILES["ranking"], "ranking")
model_results = read_csv_if_exists(FILES["model_results"], "model_results")
kfold_top100 = read_csv_if_exists(FILES["kfold_top100"], "kfold_top100")
dnn_features_results = read_csv_if_exists(FILES["dnn_features"], "dnn_features_results")
dnn_final = read_csv_if_exists(FILES["dnn_final"], "dnn_final")
clustering_results = read_csv_if_exists(FILES["clustering"], "clustering_results")
final_comparison = read_csv_if_exists(FILES["final_comparison"], "final_comparison")

if df is None:
    raise FileNotFoundError(
        "No se encontró leukemia_handcrafted_features.csv. "
        "Ubica el CSV de features en la carpeta del proyecto o define LEUKEMIA_PROJECT_DIR."
    )


## 4. Preparación del dataframe de features

El CSV ya contiene las variables extraídas por los notebooks anteriores. Para ser reproducibles se repite la limpieza mínima:

1. Separar columnas descriptivas de variables numéricas.
2. Reemplazar infinitos por NaN.
3. Imputar NaN con la mediana.
4. Eliminar columnas constantes.
5. Codificar las clases.


In [ ]:
metadata_cols = [
    "class",
    "file_name",
    "stem",
    "original_path",
    "segmented_path",
    "has_segmented",
]

numeric_feature_cols = [
    col for col in df.columns
    if col not in metadata_cols and pd.api.types.is_numeric_dtype(df[col])
]

X_raw = df[numeric_feature_cols].copy()
y = df["class"].astype(str).copy()

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
nan_before = int(X_raw.isna().sum().sum())
X_filled = X_raw.fillna(X_raw.median(numeric_only=True)).fillna(0)

constant_cols = [
    col for col in X_filled.columns
    if X_filled[col].nunique(dropna=False) <= 1
]
X_clean = X_filled.drop(columns=constant_cols)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)

prep_summary = pd.DataFrame({
    "Concepto": [
        "Muestras",
        "Columnas totales del CSV",
        "Variables numéricas candidatas",
        "Valores NaN antes de imputar",
        "Variables constantes eliminadas",
        "Variables finales usadas",
        "Clases",
    ],
    "Valor": [
        len(df),
        df.shape[1],
        len(numeric_feature_cols),
        nan_before,
        len(constant_cols),
        X_clean.shape[1],
        ", ".join(class_names),
    ],
})

display(prep_summary)


## 5. Exploración inicial del dataset

Se revisa el conteo de clases y el posible desbalance. En problemas médicos multiclase, esta distribución importa porque un accuracy alto puede ocultar fallos en clases minoritarias.


In [ ]:
class_summary = (
    y.value_counts()
    .rename_axis("class")
    .reset_index(name="n_images")
)
class_summary["proportion"] = class_summary["n_images"] / class_summary["n_images"].sum()
class_summary["class"] = pd.Categorical(class_summary["class"], CLASS_ORDER, ordered=True)
class_summary = class_summary.sort_values("class").reset_index(drop=True)

display(class_summary.assign(proportion=lambda d: d["proportion"].round(3)))

plt.figure(figsize=(7, 4))
if sns is not None:
    ax = sns.barplot(data=class_summary, x="class", y="n_images", palette="Set2")
else:
    ax = plt.bar(class_summary["class"].astype(str), class_summary["n_images"])
    ax = plt.gca()

for i, row in class_summary.iterrows():
    ax.text(i, row["n_images"] + 15, str(row["n_images"]), ha="center", va="bottom")

plt.title("Distribución de imágenes por clase")
plt.xlabel("Clase")
plt.ylabel("Número de imágenes")
plt.tight_layout()
plt.show()


In [ ]:
def resolve_image_path(row, preferred_column="original_path"):
    raw_path = row.get(preferred_column)
    if isinstance(raw_path, str) and raw_path:
        candidate = Path(raw_path)
        if candidate.exists():
            return candidate

    possible_names = []
    for col in ["file_name", "stem"]:
        value = row.get(col)
        if isinstance(value, str) and value:
            possible_names.extend([value, f"{value}.jpg", f"{value}.png", f"{value}.bmp"])

    for directory in SEARCH_DIRS:
        for name in possible_names:
            matches = list(directory.rglob(name))
            if matches:
                return matches[0]
    return None


sample_paths = []
for cls in CLASS_ORDER:
    subset = df[df["class"].astype(str) == cls]
    if len(subset) == 0:
        continue
    row = subset.iloc[0]
    path = resolve_image_path(row, "original_path")
    if path is not None:
        sample_paths.append((cls, path))

if sample_paths:
    n = len(sample_paths)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (cls, path) in zip(axes, sample_paths):
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(cls)
        ax.axis("off")
    plt.suptitle("Ejemplos visuales por clase")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown(
        "> En esta copia local no se encontraron las imágenes originales/segmentadas. "
        "El análisis usa el CSV de features ya generado; si se ejecuta en Colab con el dataset descargado, "
        "esta celda mostrará ejemplos por clase."
    ))


**Comentario exploratorio.** La clase `Benign` es la menor del conjunto, con 504 imágenes, mientras que `Early` y `Pre` superan las 960 imágenes. El desbalance no es extremo, pero sí suficiente para justificar métricas macro, porque estas asignan el mismo peso a cada clase al promediar desempeño.


## 6. Extracción y descripción de características

Los notebooks originales extrajeron variables manuales (*handcrafted features*) a partir de imágenes redimensionadas a 224x224. La extracción completa puede tardar varios minutos; por eso en esta versión final se carga el CSV ya generado.

Familias de características usadas:

- **Color:** estadísticos e histogramas en RGB, HSV y LAB.
- **Intensidad / gris:** media, desviación, cuantiles, contraste, brillo, entropía e histogramas.
- **Textura:** GLCM, LBP y resúmenes de HOG.
- **Bordes:** densidad de bordes mediante Canny.
- **Frecuencia:** DCT de baja frecuencia.
- **Morfología:** área, perímetro, circularidad, solidez, excentricidad, diámetro equivalente y momentos de Hu.
- **Ámbito de extracción:** variables globales de la imagen completa y variables `roi_` calculadas dentro de la región segmentada cuando existe máscara.

Esta decisión permite trabajar con modelos clásicos y redes densas sin usar CNN, transfer learning ni autoencoders.


In [ ]:
def categorize_feature(feature_name):
    if feature_name.startswith("global_"):
        scope = "Global"
    elif feature_name.startswith("roi_"):
        scope = "ROI segmentada"
    elif feature_name.startswith("seg_"):
        scope = "Morfología"
    else:
        scope = "Otros"

    name = feature_name.lower()
    if "rgb" in name:
        group = "Color RGB"
    elif "hsv" in name:
        group = "Color HSV"
    elif "lab" in name:
        group = "Color LAB"
    elif "gray" in name or "intensity" in name or "brightness" in name or "contrast" in name:
        group = "Intensidad / gris"
    elif "glcm" in name:
        group = "Textura GLCM"
    elif "lbp" in name:
        group = "Textura LBP"
    elif "hog" in name:
        group = "Gradientes HOG"
    elif "dct" in name:
        group = "Frecuencia DCT"
    elif "edge" in name or "canny" in name:
        group = "Bordes"
    elif name.startswith("seg_") or "mask_" in name or "hu_moment" in name:
        group = "Morfología"
    else:
        group = "Otros"
    return scope, group


feature_families = pd.DataFrame(
    [(*categorize_feature(col), col) for col in X_clean.columns],
    columns=["scope", "feature_group", "feature"],
)

family_summary = (
    feature_families
    .groupby(["scope", "feature_group"])
    .size()
    .reset_index(name="n_features")
    .sort_values(["scope", "feature_group"])
)

display(family_summary)

plt.figure(figsize=(10, 6))
plot_data = family_summary.sort_values("n_features", ascending=False)
if sns is not None:
    sns.barplot(data=plot_data, y="feature_group", x="n_features", hue="scope")
else:
    plt.barh(plot_data["feature_group"], plot_data["n_features"])
plt.title("Número de variables finales por familia")
plt.xlabel("Número de variables")
plt.ylabel("Familia")
plt.tight_layout()
plt.show()


## 7. Análisis estadístico y ranking de variables

El ranking de variables se construyó combinando tres criterios:

1. **ANOVA F-score:** detecta diferencias promedio entre clases.
2. **Mutual Information:** mide dependencia no lineal entre variable y etiqueta.
3. **Random Forest importance:** estima relevancia predictiva mediante árboles.

Las columnas constantes se eliminaron antes del ranking, porque no aportan información discriminativa.


In [ ]:
if ranking is not None:
    ranking_clean = ranking[ranking["feature"].isin(X_clean.columns)].copy()
    ranking_clean = ranking_clean.sort_values("mean_rank_score", ascending=True).reset_index(drop=True)

    top_features = ranking_clean.head(25)[[
        "feature", "anova_f", "anova_pvalue", "mutual_info", "rf_importance", "mean_rank_score"
    ]]
    display(top_features.round(5))

    top100_families = pd.DataFrame(
        [(*categorize_feature(feature), feature) for feature in ranking_clean.head(100)["feature"]],
        columns=["scope", "feature_group", "feature"],
    )
    top100_summary = (
        top100_families
        .groupby(["scope", "feature_group"])
        .size()
        .reset_index(name="n_top100")
        .sort_values("n_top100", ascending=False)
    )
    display(top100_summary)

    plt.figure(figsize=(10, 5))
    if sns is not None:
        sns.barplot(data=top100_summary, y="feature_group", x="n_top100", hue="scope")
    else:
        plt.barh(top100_summary["feature_group"], top100_summary["n_top100"])
    plt.title("Familias más frecuentes dentro del Top 100 de variables")
    plt.xlabel("Número de variables en Top 100")
    plt.ylabel("Familia")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown(
        "> No se encontró `leukemia_feature_ranking.csv`. "
        "El notebook conserva la metodología, pero no muestra ranking calculado."
    ))


**Interpretación estadística.** Las mejores variables no pertenecen a una sola familia. En el Top 25 aparecen histogramas HSV, intensidad en gris, textura GLCM, DCT y variables morfológicas como `seg_mask_area`. En el Top 100 revisado, las familias con mayor presencia fueron textura GLCM en la ROI segmentada, color HSV/LAB/RGB global y variables morfológicas.

Esto sugiere que las características manuales sí contienen información cuantificable para diferenciar las clases. No obstante, el ranking por sí solo no prueba generalización; debe complementarse con validación supervisada.


## 8. Clasificación supervisada clásica

Se compararon modelos clásicos con separación train/test estratificada y métricas macro:

- Gaussian Naive Bayes
- Decision Tree
- Random Forest
- SVM lineal
- SVM RBF
- SVM polinomial
- KNN
- Logistic Regression

Las métricas macro son importantes porque el conjunto no está perfectamente balanceado. Así se evita que la clase mayoritaria domine la evaluación.


In [ ]:
def standardize_result_columns(results_df):
    if results_df is None:
        return None

    renamed = results_df.copy()
    normalized = {col: col.strip() for col in renamed.columns}
    renamed = renamed.rename(columns=normalized)

    mapping = {}
    for col in renamed.columns:
        key = col.lower().replace(" ", "_")
        if key in ["model", "modelo", "classifier", "clasificador"]:
            mapping[col] = "Model"
        elif key in ["feature_set", "features_set", "feature_subset", "features"]:
            mapping[col] = "Feature set"
        elif key == "n_features":
            mapping[col] = "Number of input variables"
        elif key in ["accuracy", "acc"]:
            mapping[col] = "Accuracy"
        elif key in ["precision_macro", "precision_macro_mean"]:
            mapping[col] = "Precision macro"
        elif key in ["recall_macro", "recall_macro_mean"]:
            mapping[col] = "Recall macro"
        elif key in ["specificity_macro"]:
            mapping[col] = "Specificity macro"
        elif key in ["f1_macro", "f1_macro_mean"]:
            mapping[col] = "F1 macro"
        elif key == "input_type":
            mapping[col] = "Input type"

    return renamed.rename(columns=mapping)


model_results_std = standardize_result_columns(model_results)

if model_results_std is not None:
    metric_cols = ["Accuracy", "Precision macro", "Recall macro", "Specificity macro", "F1 macro"]
    for col in metric_cols:
        if col in model_results_std.columns:
            model_results_std[col] = pd.to_numeric(model_results_std[col], errors="coerce")

    display(
        model_results_std
        .sort_values("F1 macro", ascending=False)
        .reset_index(drop=True)
        .round(5)
    )

    if "Feature set" in model_results_std.columns:
        all_features_results = (
            model_results_std[model_results_std["Feature set"].astype(str).str.contains("all", case=False, na=False)]
            .sort_values("F1 macro", ascending=False)
            .reset_index(drop=True)
        )
        display(Markdown("**Resultados usando todas las features finales:**"))
        display(all_features_results.round(5))

        best_by_feature_set = (
            model_results_std
            .sort_values("F1 macro", ascending=False)
            .groupby("Feature set", as_index=False)
            .first()
            .sort_values("F1 macro", ascending=False)
        )
        display(Markdown("**Mejor modelo por subconjunto de variables:**"))
        display(best_by_feature_set.round(5))
else:
    display(Markdown("> No se encontró `model_comparison_results.csv`."))


In [ ]:
def specificity_macro_score(y_true, y_pred, labels=None):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    specificities = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    return float(np.mean(specificities))


def build_classical_models():
    return {
        "Gaussian Naive Bayes": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB(var_smoothing=5.061576888752309e-07)),
        ]),
        "Decision Tree": Pipeline([
            ("model", DecisionTreeClassifier(
                criterion="entropy",
                max_depth=15,
                min_samples_leaf=4,
                min_samples_split=9,
                class_weight="balanced",
                random_state=SEED,
            )),
        ]),
        "Random Forest": Pipeline([
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),
        "SVM Linear": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="linear", C=1, class_weight="balanced", random_state=SEED)),
        ]),
        "SVM RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=SEED)),
        ]),
        "SVM Polynomial": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="poly", degree=3, C=1, gamma="scale", class_weight="balanced", random_state=SEED)),
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5)),
        ]),
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", n_jobs=-1, random_state=SEED)),
        ]),
    }


best_classical_name = None
if model_results_std is not None and "F1 macro" in model_results_std.columns:
    best_classical_name = model_results_std.sort_values("F1 macro", ascending=False).iloc[0]["Model"]
    display(Markdown(f"**Mejor modelo supervisado clásico reportado:** `{best_classical_name}`."))

cm_image_name = "cm_svm_linear.png" if best_classical_name == "SVM Linear" else None
cm_image_path = find_file(cm_image_name) if cm_image_name else None

if cm_image_path is not None:
    display(Markdown("**Matriz de confusión del mejor modelo clásico, recuperada del notebook original:**"))
    display(IPImage(filename=str(cm_image_path)))
elif best_classical_name is not None:
    display(Markdown("**No se encontró imagen de matriz de confusión; se recalcula con el split estratificado.**"))
    models = build_classical_models()
    model = models.get(best_classical_name)
    if model is not None:
        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))

        plt.figure(figsize=(6, 5))
        if sns is not None:
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        else:
            plt.imshow(cm, cmap="Blues")
            plt.colorbar()
            plt.xticks(range(len(class_names)), class_names)
            plt.yticks(range(len(class_names)), class_names)
        plt.title(f"Matriz de confusión - {best_classical_name}")
        plt.xlabel("Predicción")
        plt.ylabel("Clase real")
        plt.tight_layout()
        plt.show()

        print(classification_report(y_test, y_pred, target_names=class_names, digits=4))


### Validación cruzada

En los notebooks previos se usó validación cruzada estratificada de 10 folds. En esta copia local no está disponible el archivo `kfold10_results_top100.csv`; por eso se conserva una tabla de referencia extraída de la salida guardada del notebook maestro. Si el CSV existe, se usa automáticamente.


In [ ]:
kfold_reference = pd.DataFrame([
    ["SVM RBF", "top100_features", 0.99171, 0.00532, 0.98882, 0.99156, 0.99004, 0.00639],
    ["SVM Linear", "top100_features", 0.99079, 0.00513, 0.98781, 0.99061, 0.98905, 0.00600],
    ["Logistic Regression", "top100_features", 0.98987, 0.00659, 0.98719, 0.98900, 0.98791, 0.00793],
    ["Random Forest", "top100_features", 0.98557, 0.00659, 0.98176, 0.98452, 0.98299, 0.00798],
    ["KNN", "top100_features", 0.98526, 0.00527, 0.98204, 0.98335, 0.98259, 0.00615],
    ["SVM Polynomial", "top100_features", 0.96407, 0.00775, 0.95872, 0.95866, 0.95844, 0.00896],
    ["Decision Tree", "top100_features", 0.96192, 0.00980, 0.95421, 0.95633, 0.95500, 0.01205],
    ["Gaussian Naive Bayes", "top100_features", 0.89895, 0.00980, 0.88555, 0.89428, 0.88759, 0.00965],
], columns=[
    "model", "feature_set", "accuracy_mean", "accuracy_std",
    "precision_macro_mean", "recall_macro_mean", "f1_macro_mean", "f1_macro_std"
])

if kfold_top100 is not None:
    display(kfold_top100.sort_values("f1_macro_mean", ascending=False).round(5))
else:
    display(kfold_reference.round(5))

best_model_kfold_reference = pd.DataFrame({
    "metric": ["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    "mean": [0.99202, 0.99033, 0.99059, 0.99037],
    "std": [0.00517, 0.00699, 0.00535, 0.00609],
})
display(Markdown("**K-Fold final reportado para el mejor modelo de la sección clásica:**"))
display(best_model_kfold_reference.round(5))

RUN_FINAL_KFOLD_RECOMPUTE = False

if RUN_FINAL_KFOLD_RECOMPUTE and best_classical_name is not None:
    models = build_classical_models()
    best_model = models[best_classical_name]
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    scoring = {
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
    }
    scores = cross_validate(best_model, X_clean, y_encoded, cv=cv, scoring=scoring, n_jobs=-1)
    recomputed = pd.DataFrame({
        "metric": list(scoring.keys()),
        "mean": [scores[f"test_{metric}"].mean() for metric in scoring.keys()],
        "std": [scores[f"test_{metric}"].std() for metric in scoring.keys()],
    })
    display(recomputed.round(5))


**Discusión de modelos clásicos.** El mejor resultado supervisado reportado fue **SVM Linear con todas las 713 features**, con accuracy 0.9985 y F1 macro 0.9981. Random Forest también fue muy competitivo, pero ligeramente por debajo. La validación cruzada sobre Top 100 mantuvo resultados altos, lo que indica que el desempeño no depende exclusivamente de usar todas las variables.


## 9. Red neuronal profunda densa

Esta sección usa una **DNN densa** con TensorFlow/Keras sobre las variables handcrafted.

Se aclara explícitamente:

- No se usa CNN.
- No se usa transfer learning.
- No se usan autoencoders.
- La entrada no son imágenes crudas, sino las 713 variables numéricas finales.

Arquitectura reportada:

| Capa | Configuración |
|---|---|
| Entrada | 713 variables |
| Dense 1 | 256 neuronas, ReLU |
| Dropout | 0.30 |
| Dense 2 | 128 neuronas, ReLU |
| Dropout | 0.30 |
| Dense 3 | 64 neuronas, ReLU |
| Salida | 4 neuronas, softmax |

Compilación: optimizador Adam, pérdida `sparse_categorical_crossentropy`, métrica `accuracy`.


In [ ]:
def clean_final_comparison(df_final):
    if df_final is None:
        return None
    cleaned = standardize_result_columns(df_final)
    if "Model" in cleaned.columns:
        cleaned = cleaned[cleaned["Model"].notna()].copy()
    for col in ["Accuracy", "Precision macro", "Recall macro", "Specificity macro", "F1 macro"]:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")
    return cleaned


final_comparison_clean = clean_final_comparison(final_comparison)
dnn_cache = None

if dnn_features_results is not None:
    dnn_cache = standardize_result_columns(dnn_features_results)
elif final_comparison_clean is not None and "Model" in final_comparison_clean.columns:
    dnn_cache = final_comparison_clean[
        final_comparison_clean["Model"].astype(str).str.fullmatch("DNN features", case=False, na=False)
    ].copy()

if dnn_cache is not None and len(dnn_cache) > 0:
    display(Markdown("**Resultados reportados para la DNN densa con features handcrafted:**"))
    display(dnn_cache.round(5))
else:
    display(Markdown("> No se encontró CSV con resultados de DNN. Se puede activar el reentrenamiento opcional."))

for image_name, title in [
    ("dnn_features_loss_curve.png", "Curva de pérdida - DNN features"),
    ("dnn_features_accuracy_curve.png", "Curva de accuracy - DNN features"),
    ("dnn_features_confusion_matrix.png", "Matriz de confusión - DNN features"),
]:
    path = find_file(image_name)
    if path is not None:
        display(Markdown(f"**{title}**"))
        display(IPImage(filename=str(path)))


In [ ]:
RUN_DNN_TRAINING = False

if RUN_DNN_TRAINING:
    try:
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers
        from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

        tf.random.set_seed(SEED)

        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
        )
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = keras.Sequential([
            layers.Input(shape=(X_train_scaled.shape[1],), name="input_features"),
            layers.Dense(256, activation="relu", name="dense_256"),
            layers.Dropout(0.30, name="dropout_1"),
            layers.Dense(128, activation="relu", name="dense_128"),
            layers.Dropout(0.30, name="dropout_2"),
            layers.Dense(64, activation="relu", name="dense_64"),
            layers.Dense(len(class_names), activation="softmax", name="output_softmax"),
        ])
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        model.summary()

        callbacks = [
            EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6, verbose=1),
        ]

        history = model.fit(
            X_train_scaled,
            y_train,
            validation_split=0.20,
            epochs=150,
            batch_size=32,
            callbacks=callbacks,
            verbose=1,
        )

        proba = model.predict(X_test_scaled)
        y_pred = np.argmax(proba, axis=1)

        dnn_metrics = pd.DataFrame([{
            "Model": "DNN features",
            "Input type": "Handcrafted features",
            "Number of input variables": X_train_scaled.shape[1],
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "Recall macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "Specificity macro": specificity_macro_score(y_test, y_pred, labels=np.arange(len(class_names))),
            "F1 macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        }])
        display(dnn_metrics.round(5))
        print(classification_report(y_test, y_pred, target_names=class_names, digits=4))
    except Exception as exc:
        print("No fue posible entrenar la DNN en este entorno:", repr(exc))
else:
    print("RUN_DNN_TRAINING = False. Se usan los resultados cacheados revisados en los notebooks originales.")


**Discusión de la DNN.** La DNN densa con features handcrafted obtuvo F1 macro 0.9930 y accuracy 0.9939. Fue competitiva frente a Random Forest, pero no superó al SVM lineal reportado como mejor modelo global. Esto sugiere que, con features manuales bien diseñadas, los modelos clásicos pueden ser igual o más efectivos que una red densa para este conjunto.


## 10. Aprendizaje no supervisado

En aprendizaje no supervisado las etiquetas no se usan para entrenar los algoritmos. Sin embargo, una vez generados los clusters, las etiquetas reales pueden usarse para evaluar si los grupos coinciden con las clases clínicas mediante métricas externas como ARI o NMI.

Métodos integrados:

- PCA para reducción de dimensionalidad.
- Visualizaciones PCA 2D y 3D.
- K-Means.
- DBSCAN.
- Clustering jerárquico aglomerativo.


In [ ]:
RUN_PCA_SUMMARY = True

if RUN_PCA_SUMMARY:
    scaler_pca = StandardScaler()
    X_scaled = scaler_pca.fit_transform(X_clean)
    pca_full = PCA(random_state=SEED)
    pca_full.fit(X_scaled)
    cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

    pca_thresholds = pd.DataFrame({
        "varianza_objetivo": ["80%", "90%", "95%"],
        "n_componentes": [
            int(np.argmax(cumulative_var >= 0.80) + 1),
            int(np.argmax(cumulative_var >= 0.90) + 1),
            int(np.argmax(cumulative_var >= 0.95) + 1),
        ],
    })
    display(pca_thresholds)

    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, len(cumulative_var) + 1), cumulative_var, linewidth=2)
    plt.axhline(0.95, color="red", linestyle="--", label="95%")
    plt.xlabel("Número de componentes")
    plt.ylabel("Varianza acumulada")
    plt.title("PCA - Varianza explicada acumulada")
    plt.legend()
    plt.tight_layout()
    plt.show()

    pca_2 = PCA(n_components=2, random_state=SEED)
    X_pca_2 = pca_2.fit_transform(X_scaled)
    pca_2_var = pca_2.explained_variance_ratio_.sum()
    display(Markdown(f"**PCA 2D explica {pca_2_var:.2%} de la varianza.**"))

    pca_plot_df = pd.DataFrame({
        "PC1": X_pca_2[:, 0],
        "PC2": X_pca_2[:, 1],
        "class": y.values,
    })
    plt.figure(figsize=(8, 6))
    if sns is not None:
        sns.scatterplot(data=pca_plot_df, x="PC1", y="PC2", hue="class", hue_order=CLASS_ORDER, s=25, alpha=0.75)
    else:
        for cls in CLASS_ORDER:
            subset = pca_plot_df[pca_plot_df["class"] == cls]
            plt.scatter(subset["PC1"], subset["PC2"], label=cls, s=25, alpha=0.75)
        plt.legend()
    plt.title("Proyección PCA 2D por clase real")
    plt.tight_layout()
    plt.show()


In [ ]:
for image_name, title in [
    ("pca_2d_visualization.png", "PCA 2D generado en Parte 4"),
    ("pca_3d_visualization.png", "PCA 3D generado en Parte 4"),
    ("kmeans_elbow_silhouette.png", "K-Means: codo y silhouette"),
    ("clustering_metrics_comparison.png", "Comparación de métricas de clustering"),
]:
    path = find_file(image_name)
    if path is not None:
        display(Markdown(f"**{title}**"))
        display(IPImage(filename=str(path)))

if clustering_results is not None:
    clustering_clean = clustering_results.copy()
    display(
        clustering_clean
        .sort_values("ARI", ascending=False, na_position="last")
        .reset_index(drop=True)
        .round(4)
    )
else:
    display(Markdown("> No se encontró `clustering_comparison.csv`."))


**Análisis crítico del no supervisado.**

El PCA mostró que se requieren 180 componentes para retener el 95% de la varianza. Las visualizaciones 2D/3D ayudan a inspeccionar estructura, pero no capturan toda la información necesaria para separar perfectamente las cuatro clases.

El mejor clustering por ARI fue **Agglomerative (K=4, ward)** con ARI 0.6833 y NMI 0.7517. K-Means con K=4 obtuvo resultados cercanos, con ARI alrededor de 0.681. DBSCAN tuvo bajo acuerdo con las etiquetas cuando se seleccionó la mejor configuración con clusters válidos, lo cual sugiere que la estructura no se ajusta bien a grupos de densidad homogénea.

En términos biológicos y visuales, `Benign` tiende a diferenciarse mejor de las clases malignas; los subtipos `Early`, `Pre` y `Pro` son más difíciles porque comparten rasgos morfológicos y de textura.


## 11. Comparación global de resultados

Se consolidan modelos clásicos, DNN densa y modelos supervisados con PCA cuando el CSV final está disponible.


In [ ]:
if final_comparison_clean is not None and len(final_comparison_clean) > 0:
    cols = [
        "Model", "Input type", "Number of input variables",
        "Accuracy", "Precision macro", "Recall macro", "Specificity macro", "F1 macro",
    ]
    available_cols = [col for col in cols if col in final_comparison_clean.columns]
    final_display = final_comparison_clean[available_cols].sort_values("F1 macro", ascending=False)
    display(final_display.round(5))

    plt.figure(figsize=(10, 5))
    top_plot = final_display.head(8).copy()
    if sns is not None:
        sns.barplot(data=top_plot, x="F1 macro", y="Model", hue="Input type", dodge=False)
    else:
        plt.barh(top_plot["Model"], top_plot["F1 macro"])
    plt.xlim(0, 1.02)
    plt.title("Comparación global - F1 macro")
    plt.xlabel("F1 macro")
    plt.ylabel("Modelo")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("> No se encontró `final_comparison_all_models.csv`; se usan las tablas previas por sección."))


**Síntesis comparativa.**

- Mejor enfoque global reportado: **SVM Linear con 713 handcrafted features**, F1 macro 0.9981.
- Mejor modelo con PCA: **SVM RBF + PCA (180 componentes)**, F1 macro 0.9943.
- DNN densa con handcrafted features: F1 macro 0.9930, competitiva pero no superior al mejor SVM.
- Aprendizaje no supervisado: útil para explorar estructura latente, pero no reemplaza a la clasificación supervisada.
- El análisis estadístico fue valioso porque mostró que color, textura, intensidad, frecuencia y morfología contienen información discriminativa medible.


## 12. Conclusiones

1. El análisis estadístico mostró que las imágenes contienen diferencias cuantificables entre clases.
2. Las features handcrafted permitieron construir modelos de clasificación de alto desempeño.
3. Los modelos clásicos, especialmente SVM y Random Forest, fueron muy competitivos cuando las variables estuvieron bien diseñadas.
4. La DNN densa permitió evaluar un enfoque neuronal sin recurrir a CNN, transfer learning ni autoencoders.
5. PCA redujo dimensionalidad manteniendo información relevante; con 180 componentes se retuvo el 95% de la varianza.
6. Los métodos no supervisados ayudaron a explorar la estructura interna del dataset, aunque no igualaron la utilidad de los modelos supervisados.
7. El pipeline podría servir como herramienta de apoyo, triaje o segunda opinión para patólogos, no como reemplazo del diagnóstico clínico.
8. Para una aplicación real sería necesario controlar iluminación, tinción, cámara, microscopio, calidad de imagen y variabilidad entre instituciones.


## 13. Limitaciones y trabajo futuro

- El dataset proviene de una fuente específica y puede reflejar condiciones particulares de adquisición.
- Existe riesgo de sobreajuste si no se valida en datos externos.
- El desempeño debe probarse con imágenes de otros hospitales, microscopios y protocolos de tinción.
- El diagnóstico definitivo requiere integración con criterios clínicos, morfología experta, citometría de flujo y otros estudios.
- Trabajo futuro: evaluar CNN y transfer learning, comparar con modelos fundacionales de visión, aplicar validación externa y estudiar explicabilidad clínica.


## 14. Referencias

- Kaggle dataset: [mehradaria/leukemia](https://www.kaggle.com/datasets/mehradaria/leukemia/data?select=Segmented)
- Repositorio asociado: [MehradAria/ALL-Subtype-Classification](https://github.com/MehradAria/ALL-Subtype-Classification)
- Paper asociado: *A Fast and Efficient CNN Model for B-ALL Diagnosis and its Subtypes Classification using Peripheral Blood Smear Images*
